In [1]:
from openbb import obb
import pandas as pd
import pytimetk as tk
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score


In [26]:
start_date = pd.to_datetime('2015-02-27')
end_date = pd.to_datetime('2024-02-27')
symbols = "SPY"
train_until=pd.to_datetime('2018-02-27')

data = obb.equity.price.historical(
    symbols,
    start_date=start_date,
    end_date=end_date,
    provider="yfinance"
).to_dataframe()

print(data.head())

              open    high     low   close     volume  dividends  \
date                                                               
2015-02-27  211.26  211.58  210.60  210.66  108076000        0.0   
2015-03-02  210.78  212.06  210.72  211.99   87491400        0.0   
2015-03-03  211.47  212.05  210.08  211.12  110325800        0.0   
2015-03-04  210.40  210.49  209.06  210.23  114497200        0.0   
2015-03-05  210.62  210.80  209.85  210.46   76873000        0.0   

            stock_splits  capital_gains  
date                                     
2015-02-27           0.0            0.0  
2015-03-02           0.0            0.0  
2015-03-03           0.0            0.0  
2015-03-04           0.0            0.0  
2015-03-05           0.0            0.0  


In [27]:
df = data.reset_index()[['date', 'open', 'high', 'low', 'close']]
df

,date,open,high,low,close
0,2015-02-27,211.26,211.58,210.60,210.66
1,2015-03-02,210.78,212.06,210.72,211.99
2,2015-03-03,211.47,212.05,210.08,211.12
3,2015-03-04,210.40,210.49,209.06,210.23
4,2015-03-05,210.62,210.80,209.85,210.46
...,...,...,...,...,...
2260,2024-02-21,495.42,497.37,493.56,497.21
2261,2024-02-22,504.01,508.49,503.02,507.50
2262,2024-02-23,509.27,510.13,507.10,507.85
2263,2024-02-26,508.30,508.75,505.86,505.99


In [28]:
df .plot_timeseries(
        date_column="date",
        value_column="close",
        title="SPY Close",
        x_lab="date",
        y_lab="close",
    )

In [29]:
# Feature Engineering

# Distance from Moving Averages
for m in [10, 20, 30, 50, 100]:
    df[f'feat_dist_from_ma_{m}'] = df['close']/df['close'].rolling(m).mean() - 1

In [30]:
# Distance from n day max/min
for m in [6, 10, 15, 20, 30, 50, 100]:
    df[f'feat_dist_from_max_{m}'] = df['close']/df['high'].rolling(m).max() - 1
    df[f'feat_dist_from_min_{m}'] = df['close']/df['low'].rolling(m).min() - 1 

In [31]:
# Price Distance
for m in [6, 10, 15, 20, 30, 50, 100]:
    df[f'feat_price_dist_{m}'] = df['close']/df['close'].shift(m) - 1

In [32]:
# Target Variable (Predict price above 20SMA in 5 days)
df['target_ma'] = df['close'].rolling(20).mean()
df['price_above_ma'] = df['close'] > df['target_ma']
df['target'] = df['price_above_ma'].astype(int).shift(-5)

In [33]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2160 entries, 100 to 2259
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    2160 non-null   datetime64[ns]
 1   open                    2160 non-null   float64       
 2   high                    2160 non-null   float64       
 3   low                     2160 non-null   float64       
 4   close                   2160 non-null   float64       
 5   feat_dist_from_ma_10    2160 non-null   float64       
 6   feat_dist_from_ma_20    2160 non-null   float64       
 7   feat_dist_from_ma_30    2160 non-null   float64       
 8   feat_dist_from_ma_50    2160 non-null   float64       
 9   feat_dist_from_ma_100   2160 non-null   float64       
 10  feat_dist_from_max_6    2160 non-null   float64       
 11  feat_dist_from_min_6    2160 non-null   float64       
 12  feat_dist_from_max_10   2160 non-null   float64    

In [34]:
feat_cols = [col for col in df.columns if 'feat' in col]
train_until = train_until

x_train = df[df['date'] <= train_until][feat_cols]
y_train = df[df['date'] <= train_until]['target']

x_test = df[df['date'] > train_until][feat_cols]
y_test = df[df['date'] > train_until]['target']

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2160 entries, 100 to 2259
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    2160 non-null   datetime64[ns]
 1   open                    2160 non-null   float64       
 2   high                    2160 non-null   float64       
 3   low                     2160 non-null   float64       
 4   close                   2160 non-null   float64       
 5   feat_dist_from_ma_10    2160 non-null   float64       
 6   feat_dist_from_ma_20    2160 non-null   float64       
 7   feat_dist_from_ma_30    2160 non-null   float64       
 8   feat_dist_from_ma_50    2160 non-null   float64       
 9   feat_dist_from_ma_100   2160 non-null   float64       
 10  feat_dist_from_max_6    2160 non-null   float64       
 11  feat_dist_from_min_6    2160 non-null   float64       
 12  feat_dist_from_max_10   2160 non-null   float64    

In [35]:
# Train Model

clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42,
    class_weight='balanced'
)

clf.fit(x_train, y_train)

y_train_pred = clf.predict(x_train)
y_test_pred = clf.predict(x_test)

print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred)}")
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred)}")

print(f"Train Precision: {precision_score(y_train, y_train_pred)}")
print(f"Test Precision: {precision_score(y_test, y_test_pred)}")

print(f"Train ROC AUC: {roc_auc_score(y_train, clf.predict_proba(x_train)[:, 1])}")
print(f"Test ROC AUC: {roc_auc_score(y_test, clf.predict_proba(x_test)[:, 1])}")

Train Accuracy: 0.7911585365853658
Test Accuracy: 0.7300531914893617
Train Precision: 0.9054054054054054
Test Precision: 0.8669154228855721
Train ROC AUC: 0.8834832763863201
Test ROC AUC: 0.793439506055719


In [38]:
# Visualize

df_test = df[df['date'] > train_until].reset_index(drop=True)
df_test['pred_prob'] = clf.predict_proba(x_test)[:, 1]
df_test['pred'] = df_test['pred_prob'] > 0.5

fig = df_test \
    .plot_timeseries(
        date_column="date",
        value_column="close",
        title=f"{symbols} Price with Predicted Patterns",
        x_lab="date",
        y_lab="close",
    )

fig.add_trace(
    go.Line(
        x=df_test['date'],
        y=df_test['target_ma'],
        name="Target 20SMA"
    )
)

df_pattern = (
    df_test[df_test['pred']]
        .groupby((~df_test['pred']).cumsum())['date']
        .agg(['first', 'last'])
)

for idx, row in df_pattern.iterrows():
    fig.add_vrect(
        x0=row['first'],
        x1=row['last'],
        line_width=0,
        fillcolor="green",
        opacity=0.2,
    )
    
fig.update_layout(
    width = 800,
    height = 600,
    xaxis_rangeslider_visible=True,
)

fig.show()
